# Lab 09-02 — Recursive tree build with LLM summarization (RAPTOR step 2)

**Track 09 · RAPTOR** — how we turn a flat chunk partition into the summary tree RAPTOR actually queries.

Lab 01 embedded 24 passages and clustered them with a recursive GMM, producing a flat partition where every chunk lives in exactly one cluster. That partition is raw material. This lab turns it into the artifact RAPTOR actually queries: a *tree* where the leaves are the raw chunks and every internal node is a short LLM summary of the cluster of chunks beneath it. The build lives in `tools.raptor.build_tree`.

```text
24 passages (rag-mini-wikipedia, deterministic head)
  -> BGE embeddings (local, cpu)
  -> recursive GMM clusters (from lab 01's algorithm)
  -> LLM summary per cluster (qwen2.5-coder:7b, one call per node)
  -> recurse until one root remains
  -> verification gate (--verify)
```

Unlike lab 01, this lab *does* call the LLM: one summary call per internal node, far cheaper than re-reading every chunk at query time. Two properties make the tree worth building: **coverage** (every chunk in exactly one leaf) and **compression** (query navigates summaries, descends only into the relevant cluster).


## Setup

This notebook mirrors `curriculum/09-raptor/02-recursive-tree.py` exactly — the same verified code, split into cells. Two prerequisites must hold before it will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM every summary call goes to (`llms/ollama.py` talks to it through `langchain-ollama`). Fully local: no API key, no quota. If the server is not up, every summarization call fails and the tree never materializes.
- **The corpus on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet`, already fetched by the repo's manifest-verified fetchers.

The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. From the terminal the lab runs as:

```bash
python curriculum/09-raptor/02-recursive-tree.py          # run + demo
python curriculum/09-raptor/02-recursive-tree.py --verify # verification gate
```

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   langchain-ollama -> the Ollama chat backend behind llms/ollama.py
#   langchain-huggingface -> sentence-transformer embedding bridge
#   pandas           -> read the rag-mini-wikipedia parquet
#   scikit-learn     -> the GaussianMixture behind recursive GMM clustering
%pip install langchain-ollama langchain-huggingface pandas scikit-learn


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd  # noqa: E402

from embeddings.bge import BGEEmbedding  # noqa: E402
from llms.ollama import OllamaLLM  # noqa: E402
from tools.raptor import build_tree, collect_chunk_ids  # noqa: E402


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 24` takes the **deterministic head** of rag-mini-wikipedia — and because each cluster costs one LLM summarization call, this number shapes the runtime. `MAX_CLUSTER_SIZE = 8` caps how many chunks a single summary node may cover; smaller values produce a taller tree with more summaries. `BGE_DEVICE = 'cpu'` because Ollama holds most of the VRAM. `ROOT_PREVIEW` and `MID_PREVIEW` only control how many characters of the demo output to print.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 24  # deterministic head; the tree costs one LLM call per cluster
MAX_CLUSTER_SIZE = 8  # max chunks a single summary node may cover
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM
ROOT_PREVIEW = 200  # characters of the root summary to print
MID_PREVIEW = 160  # characters of the mid-level summary to print


## 2. Load — first N passages of rag-mini-wikipedia

`passages.parquet` is a plain table with a `passage` column; `head(n)` keeps the first `n` rows so every run works on the same corpus slice. Each passage is loaded **whole**: the clustering and summarization happen at the tree level, not here.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages of the rag-mini-wikipedia corpus
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment — embed, cluster, summarize recursively until one root remains

`build_tree` (in `tools/raptor.py`) does the heavy lifting in four steps:

- **Level 0** — every chunk becomes a leaf node carrying its own text.
- **Cluster** — the recursive GMM from lab 01 groups leaf embeddings into clusters, each at most `MAX_CLUSTER_SIZE` chunks.
- **Summarize** — the local LLM (`qwen2.5-coder:7b` on `localhost:11434`) reads each cluster and produces a 2-3 sentence summary; that summary becomes a parent node whose children are the cluster's leaves.
- **Recurse** — the summaries become the new level to cluster and summarize, until one node (the root) remains.

Two properties make the tree useful instead of a glorified flat list: **coverage** (every chunk appears in exactly one leaf, so no information is dropped) and **compression** (a query can navigate the few summary nodes at the top and only descend into the cluster that actually matters). One LLM call per internal node.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed, cluster, summarize recursively until one root remains
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    llm = OllamaLLM()  # local qwen2.5-coder:7b; temperature 0
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device=BGE_DEVICE)

    t0 = time.perf_counter()
    info = build_tree(
        passages,
        embedder,
        llm,
        max_cluster_size=MAX_CLUSTER_SIZE,
        progress=lambda level, count: print(
            f"  built level {level}: {count} node(s)", end="\r", flush=True
        ),
    )
    build_s = time.perf_counter() - t0
    print(" " * 40, end="\r")

    mid_nodes = [
        node
        for node in _walk(info["tree"])
        if node["level"] == 1 and node["children"]
    ]
    return {
        "passages": passages,
        "tree": info["tree"],
        "levels": info["levels"],
        "leaves": info["leaves"],
        "llm_calls": info["llm_calls"],
        "build_s": build_s,
        "node_counts": _node_counts_per_level(info["tree"]),
        "root_text": info["tree"]["text"],
        "mid_text": mid_nodes[0]["text"] if mid_nodes else "",
    }


def _walk(node: dict):
    """Depth-first iterator over every node of the tree."""
    yield node
    for child in node["children"]:
        yield from _walk(child)


def _node_counts_per_level(root: dict) -> dict[int, int]:
    """Number of nodes per level (level 0 = leaves)."""
    counts: dict[int, int] = {}
    for node in _walk(root):
        counts[node["level"]] = counts.get(node["level"], 0) + 1
    return dict(sorted(counts.items()))


## 4. Demo

The demo prints the artifact from four angles: nodes per level (showing the tree's shape), a preview of the root summary, a preview of one mid-level summary, and the takeaway. The takeaway is the point: the tree is a lossy but *navigable* index. Level 0 keeps every chunk verbatim, and each higher level compresses a cluster into a few sentences. Retrieval walks the summaries and only expands the cluster that matches the question — lab 03 compares that walk against scanning every chunk flat.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the tree artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 09-02 — Recursive tree build with LLM summarization")
    print(f"{exp['leaves']} leaves, {exp['levels']} levels, "
          f"{exp['llm_calls']} LLM calls in {exp['build_s']:.1f}s")
    print("=" * 66)

    print(f"\n[1] Nodes per level:")
    for level, count in exp["node_counts"].items():
        role = "root" if level == exp["levels"] else (
            "leaves" if level == 0 else "summaries")
        print(f"    level {level}: {count:3d} node(s)  [{role}]")

    print(f"\n[2] Root summary (first {ROOT_PREVIEW} chars):")
    print(f"    {exp['root_text'][:ROOT_PREVIEW]}")

    print(f"\n[3] One mid-level summary (first {MID_PREVIEW} chars):")
    print(f"    {exp['mid_text'][:MID_PREVIEW] or '(no mid-level node)'}")

    print(f"\n[4] Takeaway")
    print("    The tree is a lossy but navigable index: level 0 keeps every")
    print("    chunk verbatim, and each higher level compresses a cluster")
    print("    into a few sentences. Retrieval walks the summaries and only")
    print("    expands the cluster that matches the question — lab 03")
    print("    compares that walk against scanning every chunk flat.")


## 5. Verification gate

The lab ships a `--verify` gate: hard checks the tree must clear — at least 2 levels, root exists with non-empty text, number of leaves matches the passage count, every chunk covered exactly once across leaves, every internal node has non-empty summary text, and total LLM calls stayed under 40. The gate turns "the lab ran" into "the lab ran *correctly*" — the same discipline every lab in this repo applies.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    root = exp["tree"]
    covered = sorted(collect_chunk_ids(root))

    checks.append((f"tree has >= 2 levels (got {exp['levels']})",
                   exp["levels"] >= 2))
    checks.append(("root exists and has non-empty text",
                   bool(root.get("text", "").strip())))
    checks.append((f"number of leaves == {len(exp['passages'])} "
                   f"(got {exp['leaves']})",
                   exp["leaves"] == len(exp["passages"])))
    checks.append(("every chunk covered exactly once across leaves",
                   covered == list(range(len(exp["passages"])))))
    checks.append(("every non-leaf node has non-empty summary text",
                   all(node["text"].strip()
                       for node in _walk(root) if node["children"])))
    checks.append((f"llm_calls <= 40 (got {exp['llm_calls']})",
                   exp["llm_calls"] <= 40))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Expect a few minutes: BGE embedding is fast, but the recursive GMM clustering plus a handful of local LLM summary calls (one per internal node) add up. No downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The tree, its level structure, root and mid-level summaries — the lossy but navigable index the corpus was compressed into.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, the build pipeline misbehaved: check that Ollama is serving `qwen2.5-coder:7b` and that the corpus parquet is intact.


In [ ]:
verify_gate(exp)
